<a href="https://colab.research.google.com/github/ambreenraheem/PGD_generative_AI_NED/blob/main/AI_Career_Counselor_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ambreen Abdul Raheem
(Power BI Data Analyst-Upwork Freelancer)\
MidTerm Project: AI Career Counselor Agent

In [ ]:
import os
import requests
import chainlit as cl
from dotenv import load_dotenv

# Load environment variables
load_dotenv()
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")
GEMINI_URL = "https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent"

headers = {
    "Content-Type": "application/json",
    "x-goog-api-key": GEMINI_API_KEY
}

chat_history = []

SYSTEM_PROMPT = (
    "You are a professional, empathetic, and thorough global career counselor for students. "
    "When a student asks about careers or professions, respond with a structured, actionable plan. "
    "Include: 1) Overview, 2) Responsibilities, 3) Education/Pathways, 4) Skills, "
    "5) Salary & Outlook (as estimates), 6) Related roles, 7) Learning resources, "
    "8) Preparation by school stage, 9) Interview/CV tips, 10) 6–12 month personalized plan. "
    "Ask one clarifying question if the user's query is vague. "
    "Tone should be encouraging and clear. Tailor advice for the student's age & location if given."
)

def ensure_system_prompt():
    if not chat_history:
        chat_history.append({
            "role": "user",
            "parts": [{"text": f"Instruction: {SYSTEM_PROMPT}"}]
        })

def build_user_message(user_input, student_context):
    if student_context.strip():
        return {
            "role": "user",
            "parts": [{"text": f"Student context: {student_context.strip()}\nQuestion: {user_input.strip()}"}]
        }
    else:
        return {"role": "user", "parts": [{"text": user_input.strip()}]}

def call_gemini(user_input, student_context=""):
    ensure_system_prompt()
    chat_history.append(build_user_message(user_input, student_context))

    body = {
        "contents": chat_history,
        "generationConfig": {
            "temperature": 0.3,
            "topK": 1,
            "topP": 0.95,
            "maxOutputTokens": 2048
        }
    }

    try:
        response = requests.post(GEMINI_URL, headers=headers, json=body, timeout=30)
    except Exception as e:
        return f"**Error:** Request failed — {str(e)}"

    if response.status_code == 200:
        try:
            result = response.json()
            parts = result["candidates"][0]["content"].get("parts", [])
            reply = parts[0]["text"] if parts else "_(No content returned — possibly cut off due to token limit)_"
        except Exception as e:
            reply = f"**Error parsing model reply:** {str(e)}"

        chat_history.append({"role": "model", "parts": [{"text": reply}]})
        return reply
    else:
        return f"**Error {response.status_code}:** {response.text}"

@cl.on_message
async def main(message: cl.Message):
    # Extract user input
    if "|" in message.content:
        user_input, student_context = message.content.split("|", 1)
    else:
        user_input, student_context = message.content, ""

    # Get Gemini response
    reply = call_gemini(user_input, student_context)
    await cl.Message(content=reply).send()

Dockerfile\
for deploying on HuggingFace

In [ ]:
FROM python:3.10-slim

# Avoid Python .pyc files and enable unbuffered output
ENV PYTHONDONTWRITEBYTECODE=1
ENV PYTHONUNBUFFERED=1

# Set working directory
WORKDIR /app

# Copy and install dependencies
COPY requirements.txt .
RUN pip install --no-cache-dir --upgrade pip && \
    pip install --no-cache-dir -r requirements.txt && \
    pip install --no-cache-dir chainlit python-dotenv requests

# Copy all project files
COPY . /app

# Create Chainlit folders and files
RUN mkdir -p /app/.files /app/.chainlit && \
    touch /app/chainlit.md && \
    chmod -R 777 /app/.files /app/.chainlit /app/chainlit.md

# Set environment variables for Chainlit
ENV CHAINLIT_UI=True
ENV CHAINLIT_BROWSER_AUTO_OPEN=false

# Expose port
EXPOSE 7860

# Start Chainlit app using Python module to avoid PATH issues
ENTRYPOINT ["python", "-m", "chainlit", "run", "app.py", "--host", "0.0.0.0", "--port", "7860"]

requirements.txt\
for deploying on HuggingFace

In [ ]:
python-dotenv
chainlit
requests
websockets
Docker